# API keys, environment variables, and validated configuration

## Learning goals

- Keep API keys outside source code and notebook output
- Load local values from `.env` without overriding existing shell variables
- Understand why raw environment variables need parsing and validation
- Use Pydantic to create typed, constrained configuration models
- Build a reusable `ProviderConfig` without exposing credentials


## Configuration is a security boundary

API keys are credentials, not ordinary application data. Store real values in `.env` for local development and commit only `.env.example`, which documents the required variable names without containing secrets.

Three rules will keep us out of the most common trouble:

1. Never place a real key directly in Python code, Markdown, screenshots, or notebook output.
2. Never print the value returned by `os.getenv("OPENAI_API_KEY")`. Check only whether a value exists.
3. Treat environment variables as untrusted strings until they have been parsed and validated.

> If a real key is accidentally committed or shared, remove it from the provider account and create a new one. Deleting it from the latest Git commit is not enough because it may remain in history.


## Install the dependencies with uv

When adding these packages to a new project, run:

```text
uv add python-dotenv pydantic
```

This course repository already declares its dependencies, so after cloning it use:

```text
uv sync
```

`python-dotenv` copies values from `.env` into the process environment. Pydantic turns those raw values into a typed, validated Python object. These commands are the same in PowerShell, Command Prompt, macOS, and Linux.


## Mental model

```mermaid
flowchart LR
  A[Shell environment] --> C[os.environ]
  B[Local .env file] -->|load_dotenv override=False| C
  C --> D[Raw configuration strings]
  D -->|Pydantic parses and validates| E[ProviderConfig]
  D -. invalid value .-> F[ValidationError]
  E --> G[Provider client]
  E --> H[Secret-safe summary and logs]
```

The diagram depicts a configuration pipeline rather than direct access to `.env` throughout the application:

1. **Sources:** deployment platforms, CI systems, or your terminal can provide variables directly. During local development, `python-dotenv` can fill in missing values from `.env`.
2. **Raw boundary:** `os.environ` stores every value as text. For example, `MAX_AGENT_STEPS=8` arrives as the string `"8"`, not the integer `8`. A present value can still be malformed.
3. **Validation boundary:** Pydantic converts compatible strings to the declared types, checks constraints, applies defaults, and raises a readable `ValidationError` for invalid configuration.
4. **Application boundary:** the rest of the program receives one validated `ProviderConfig` object. Provider clients do not need to repeatedly read and interpret environment variables.
5. **Logging boundary:** diagnostics expose safe facts such as provider name, model name, and whether a key exists—not the key itself.

> `override=False` preserves a value already supplied by the shell or deployment platform. The local `.env` file fills gaps instead of silently replacing higher-priority runtime configuration.


## Load `.env` safely

Run this after selecting the **Agentic AI Engineering** kernel. The code reports only whether the file and key were found; it never displays the credential.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parents[1]
#dotenv_path = Path.cwd() / ".env"
dotenv_path = project_root / ".env"
print("Loading .env file from:", dotenv_path)
dotenv_loaded = load_dotenv(dotenv_path=dotenv_path, override=False)

print("Expected .env location:", dotenv_path)
print(".env file found:", dotenv_loaded)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))


## What Pydantic adds

A normal Python class can hold configuration, but it does not automatically parse external input or enforce rules. A Pydantic model declares the shape of valid data using familiar type hints.

Pydantic is useful here for four reasons:

- **Type conversion:** it can turn the environment string `"8"` into the integer `8`.
- **Constraints:** `Field(ge=1, le=100)` rejects values outside an allowed range.
- **Defaults:** one well-defined place supplies safe fallback values.
- **Clear failures:** invalid configuration fails early with field-specific errors instead of causing a confusing failure during an agent run.

The following model validates two agent limits. Notice that `os.getenv` supplies strings, while the resulting attributes are integers.


In [ ]:
from pydantic import BaseModel, Field, ValidationError


class AgentLimits(BaseModel):
    max_steps: int = Field(default=8, ge=1, le=100)
    max_tool_calls: int = Field(default=10, ge=0, le=100)


limits = AgentLimits(
    max_steps=os.getenv("MAX_AGENT_STEPS", "8"),
    #max_steps=int("-1"), # Uncommenting this line will throw Validation Error
    max_tool_calls=os.getenv("MAX_TOOL_CALLS", "10"),
)

print(limits)
print("max_steps type:", type(limits.max_steps).__name__)


### A readable failure is a feature

This example intentionally supplies an invalid value but catches the exception, so the notebook continues running. In a real application startup path, we would usually let invalid configuration stop the process.


In [ ]:
try:
    AgentLimits(max_steps="zero")
except ValidationError as error:
    for issue in error.errors(include_url=False):
        print({
            "field": issue["loc"],
            "message": issue["msg"],
            "input": issue["input"],
        })


## Purpose of `ProviderConfig`

`ProviderConfig` is the single, validated description of how the application should connect to an AI provider. It sits between raw environment variables and provider-specific client code.

Its responsibilities are deliberately narrow:

- identify the provider and model selected for this run;
- hold an optional custom base URL for compatible gateways or local services;
- protect the API key with Pydantic's `SecretStr`;
- reject empty or unsupported values before a network request is attempted;
- provide a safe summary for debugging and logs.

It does **not** call the model, choose tools, retry requests, or implement the agent loop. Keeping configuration separate from behavior makes it easier to test, switch providers, and diagnose startup problems.

`SecretStr` masks its value when represented. We also set `repr=False`, which omits the field from the model's normal representation. Even with these safeguards, application code should never deliberately print or log `get_secret_value()`.


In [ ]:
from typing import Literal

from pydantic import BaseModel, Field, SecretStr


class ProviderConfig(BaseModel):
    provider: Literal["openai", "anthropic", "gemini", "groq", "openrouter"]
    model: str = Field(min_length=1)
    api_key: SecretStr | None = Field(default=None, repr=False)
    base_url: str | None = None

    @property
    def is_configured(self) -> bool:
        return self.api_key is not None

    def safe_summary(self) -> dict[str, str | bool | None]:
        return {
            "provider": self.provider,
            "model": self.model,
            "base_url": self.base_url,
            "has_api_key": self.is_configured,
        }


config = ProviderConfig(
    provider="openai",
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    api_key=os.getenv("OPENAI_API_KEY") or None,
    base_url=os.getenv("OPENAI_BASE_URL") or None,
)

print(config)
print(config.safe_summary())


### Reading the result

- `provider` is restricted by `Literal`, so a typo such as `"opneai"` is rejected.
- `model` must contain at least one character.
- `api_key` is either a protected `SecretStr` or `None`; it is never reduced to a boolean inside the configuration object, so a future client can receive the credential when needed.
- `is_configured` answers the safe operational question: *is a key available?*
- `safe_summary()` is intentionally allow-listed. Adding a new secret field later will not automatically leak it into logs.

> The original `has_key: bool` approach was safe to display, but it discarded the credential. The improved model safely retains the key for the provider client while exposing only a boolean in diagnostics.


## Common failure cases

| Symptom | Likely cause | Improvement |
|---|---|---|
| `.env file found: False` | The file is missing or the notebook started from another working directory | Create `.env` at the project root and inspect `os.getcwd()` |
| Key reports `False` | The variable is absent or empty | Add the value locally without printing it |
| `ModuleNotFoundError: pydantic` | The wrong kernel is selected or dependencies were not synced | Run `uv sync` and select **Agentic AI Engineering** |
| `ValidationError` during startup | A value has the wrong type, unsupported name, or invalid range | Correct the source value; do not silently ignore the error |
| A shell value appears unchanged after editing `.env` | `override=False` gives the existing shell value priority | Update or unset the shell variable, or restart the kernel |

Notebook kernels are long-running processes. After changing environment variables, restart the kernel and rerun the cells so the process receives a clean configuration state.


## Exercise

Create a second `ProviderConfig` for Anthropic using `ANTHROPIC_API_KEY` and a model name of your choice. Then:

1. Display only `safe_summary()`.
2. Confirm that an unsupported provider name produces a `ValidationError`.
3. Confirm that neither the model representation nor your summary contains the API key.
4. Explain why provider configuration should be validated before constructing a provider client.


## Summary

`.env` is a local source of configuration, `python-dotenv` loads missing values into the process environment, and Pydantic converts those raw strings into trustworthy Python objects. `ProviderConfig` gives the application one validated, provider-neutral boundary while keeping credentials out of routine output. Validate configuration once at startup, pass the resulting object inward, and log only explicitly safe fields.
